# RawGrad — MNIST Classification

Training a neural network from scratch using NumPy only.

This experiment demonstrates:
- MNIST dataset loading
- Pixel normalization
- Dense neural network architecture
- Mini-batch training
- Forward propagation
- Backpropagation
- SGD optimization
- Training and test evaluation

In [2]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.activations import ReLU, Softmax
from rawgrad.classification_losses import CategoricalCrossEntropy
from rawgrad.layers import Dense
from rawgrad.model import Sequential
from rawgrad.mnist import load_mnist
from rawgrad.optimizers import SGD

In [3]:
X_train, y_train, X_test, y_test = load_mnist()

print("Training data:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test data:", X_test.shape)
print("Test labels:", y_test.shape)

Training data: (60000, 784)
Training labels: (60000,)
Test data: (10000, 784)
Test labels: (10000,)


In [4]:
print("Pixel minimum:", X_train.min())
print("Pixel maximum:", X_train.max())
print("Classes:", np.unique(y_train))

Pixel minimum: 0.0
Pixel maximum: 1.0
Classes: [0 1 2 3 4 5 6 7 8 9]


In [5]:
def one_hot(labels, num_classes=10):
    result = np.zeros((len(labels), num_classes))
    result[np.arange(len(labels)), labels] = 1.0
    return result


y_train_one_hot = one_hot(y_train)
y_test_one_hot = one_hot(y_test)

print(y_train_one_hot.shape)
print(y_test_one_hot.shape)

(60000, 10)
(10000, 10)


In [6]:
X_train_small = X_train[:5000]
y_train_small = y_train_one_hot[:5000]

X_test_small = X_test[:1000]
y_test_small = y_test_one_hot[:1000]

print("Training subset:", X_train_small.shape)
print("Test subset:", X_test_small.shape)

Training subset: (5000, 784)
Test subset: (1000, 784)


In [7]:
np.random.seed(42)

model = Sequential([
    Dense(784, 128, initialization="xavier"),
    ReLU(),
    Dense(128, 10, initialization="xavier"),
    Softmax(),
])

loss_function = CategoricalCrossEntropy()
optimizer = SGD(learning_rate=0.03)

In [8]:

import numpy as np
from rawgrad.mnist import load_mnist
from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.classification_losses import CategoricalCrossEntropy
from rawgrad.model import Sequential
from rawgrad.optimizers import SGD
#reproducibility
np.random.seed(42)
# Load MNIST dataset
print("Loading MNIST dataset...")
X_train, y_train, X_test, y_test = load_mnist()
# Use a subset first so training is manageable.
train_indices = np.random.permutation(len(X_train))[:5000]
test_indices = np.random.permutation(len(X_test))[:1000]
X_train = X_train[train_indices]
y_train = y_train[train_indices]
X_test = X_test[test_indices]
y_test = y_test[test_indices]
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Input features:", X_train.shape[1])
print("Number of classes:", len(np.unique(y_train)))
#  Build the neural network
model = Sequential([
    Dense(784, 64, initialization="he"),
    ReLU(),
    Dense(64, 10, initialization="xavier"),
    Softmax(),
])
loss_function = CategoricalCrossEntropy()
optimizer = SGD(learning_rate=0.03)
#train config
epochs = 5
batch_size = 64
# Train the network using mini-batches
print("\nStarting MNIST training...")
for epoch in range(epochs):
    # Shuffle the training examples each epoch.
    shuffled_indices = np.random.permutation(len(X_train))
    X_shuffled = X_train[shuffled_indices]
    y_shuffled = y_train[shuffled_indices]
    total_loss = 0.0
    correct_predictions = 0
    samples_seen = 0
    for start in range(0, len(X_train), batch_size):
        end = start + batch_size
        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]
        # Convert integer labels into one-hot vectors.
        y_batch_one_hot = np.eye(10)[y_batch]
        # Forward propagation.
        predictions = model.forward(X_batch)
        # Calculate categorical cross-entropy
        loss = loss_function.forward(
            predictions,
            y_batch_one_hot,
        )
        # Backpropagation.
        gradient = loss_function.backward()
        model.backward(gradient)
        # Update weights and biases.
        optimizer.step(model)
        # Track training statistics.
        batch_samples = len(X_batch)
        total_loss += loss * batch_samples
        predicted_classes = np.argmax(
            predictions,
            axis=1,
        )
        correct_predictions += np.sum(
            predicted_classes == y_batch
        )
        samples_seen += batch_samples
    epoch_loss = total_loss / samples_seen
    epoch_accuracy = (
        correct_predictions / samples_seen
    ) * 100
    print(
        f"Epoch {epoch + 1:02d}/{epochs} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )
#evaluate
print("\nEvaluating on test data...")
test_predictions = model.forward(X_test)
y_test_one_hot = np.eye(10)[y_test]
test_loss = loss_function.forward(
    test_predictions,
    y_test_one_hot,
)
predicted_classes = np.argmax(
    test_predictions,
    axis=1,
)
test_accuracy = np.mean(
    predicted_classes == y_test
) * 100
#display eval
print("RAWGRAD: MNIST CLASSIFICATION RESULTS")
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Training epochs  : {epochs}")
print(f"Batch size       : {batch_size}")
print(f"Test loss        : {test_loss:.4f}")
print(f"Test accuracy    : {test_accuracy:.2f}%")
print("\nSample predictions:")
for index in range(10):
    actual = y_test[index]
    predicted = predicted_classes[index]
    confidence = test_predictions[index, predicted] * 100
    print(
        f"Actual: {actual} | "
        f"Predicted: {predicted} | "
        f"Confidence: {confidence:.2f}%"
    )
#verify output
assert test_predictions.shape == (1000, 10)
assert np.isfinite(test_predictions).all()
assert np.allclose(
    test_predictions.sum(axis=1),
    1.0,
)
print("\nAll output checks passed.")
print("MNIST CLASSIFICATION PIPELINE: PASS")


Loading MNIST dataset...
Already downloaded: train-images-idx3-ubyte.gz
Already downloaded: train-labels-idx1-ubyte.gz
Already downloaded: t10k-images-idx3-ubyte.gz
Already downloaded: t10k-labels-idx1-ubyte.gz
Training samples: 5000
Testing samples: 1000
Input features: 784
Number of classes: 10

Starting MNIST training...
Epoch 01/5 | Loss: 1.6109 | Accuracy: 56.42%
Epoch 02/5 | Loss: 0.8327 | Accuracy: 81.80%
Epoch 03/5 | Loss: 0.6071 | Accuracy: 85.66%
Epoch 04/5 | Loss: 0.5075 | Accuracy: 87.42%
Epoch 05/5 | Loss: 0.4501 | Accuracy: 88.40%

Evaluating on test data...
RAWGRAD: MNIST CLASSIFICATION RESULTS
Training samples : 5000
Testing samples  : 1000
Training epochs  : 5
Batch size       : 64
Test loss        : 0.4263
Test accuracy    : 87.40%

Sample predictions:
Actual: 6 | Predicted: 2 | Confidence: 44.71%
Actual: 9 | Predicted: 4 | Confidence: 81.84%
Actual: 8 | Predicted: 8 | Confidence: 70.67%
Actual: 5 | Predicted: 5 | Confidence: 71.45%
Actual: 9 | Predicted: 9 | Confiden